# Span Results

Runs all available analyses directly from Google Sheets and displays per-test reports and a domain chart.

**To use:**
1. Run the **Setup** cell — installs planars if needed and asks for Google permission (once per session)
2. Set `DRIVE_FOLDER_PATH` in the **Configure** cell to your planars Drive folder
3. Run **Load manifest**
4. Run any per-analysis report cells, or run all with **Runtime → Run all cells**
5. Run **Domain chart** to see all spans visualised

In [ ]:
# Setup — run once per session
import subprocess, sys, importlib, importlib.metadata

# Install planars if not already available
try:
    _ver = importlib.metadata.version('planars')
except importlib.metadata.PackageNotFoundError:
    print("Installing planars (first time this session)...")
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--pre',
         'planars', 'gspread', 'google-auth'])
    importlib.invalidate_caches()
    _ver = importlib.metadata.version('planars')
print(f"planars {_ver} ✓")

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from googleapiclient.discovery import build
creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
])
gc = gspread.authorize(creds)
drive_svc = build('drive', 'v3', credentials=creds)

print('Setup complete.')

In [ ]:
# Configure — set the Drive folder ID for your planars folder
# Find it in the folder URL: drive.google.com/drive/folders/<THIS_PART>
DRIVE_FOLDER_ID = 'your-folder-id-here'

In [ ]:
# Load manifest from Drive
import json

results = drive_svc.files().list(
    q=f"'{DRIVE_FOLDER_ID}' in parents and name contains 'manifest_' and trashed=false",
    fields="files(id, name)"
).execute()
if not results['files']:
    raise FileNotFoundError(
        f'No manifest file found in Drive folder: {DRIVE_FOLDER_ID}\n'
        'Please check that DRIVE_FOLDER_ID is correct.'
    )

manifest = {}
for f in results['files']:
    lang_id = f['name'].replace('manifest_', '').replace('.json', '')
    content = drive_svc.files().get_media(fileId=f['id']).execute()
    manifest[lang_id] = json.loads(content)
    print(f'Loaded manifest for: {lang_id}')

In [ ]:
# Helper — run once after loading the manifest
from planars.io import load_filled_sheet

def show_class_reports(gc, manifest, class_name, derive_fn, format_fn):
    """Print format_result() output for every construction of class_name across all languages."""
    found_any = False
    for lang_id, lang_data in manifest.items():
        sheet_info = lang_data.get('sheets', {}).get(class_name)
        if not sheet_info:
            continue
        try:
            ss = gc.open_by_key(sheet_info['spreadsheet_id'])
        except Exception as e:
            print(f'  WARNING: could not open {class_name} sheet for {lang_id}: {e}')
            continue
        construction_params = sheet_info.get('construction_params', {})
        for construction in sheet_info['constructions']:
            found_any = True
            sep = '\u2500' * 60
            print(f'\n{sep}')
            print(f'  {class_name}  [{lang_id}]  —  {construction}')
            print(sep)
            try:
                ws = ss.worksheet(construction)
            except Exception:
                print(f'  WARNING: tab not found')
                continue
            required = set(construction_params.get(construction, {}).get('param_names', []))
            loaded = load_filled_sheet(ws, required_params=required, strict=False)
            result = derive_fn(_data=loaded, strict=False)
            print(format_fn(result))
    if not found_any:
        print(f'No {class_name} sheets found in manifest.')

print('Helper defined.')

---
## Ciscategorial

For each position, tests whether elements combine with V but not with N or A.  
Reports strict and loose spans for complete and partial qualification.

In [ ]:
from planars import ciscategorial as _cisc
show_class_reports(gc, manifest, 'ciscategorial', _cisc.derive_v_ciscategorial_fractures, _cisc.format_result)

---
## Subspan Repetition

Tests repetition across five span categories (fillable, widescope left/right, narrowscope left/right).  
Each category yields strict and loose spans for complete and partial qualification (20 spans total).

In [ ]:
from planars import subspanrepetition as _subspan
show_class_reports(gc, manifest, 'subspanrepetition', _subspan.derive_subspanrepetition_spans, _subspan.format_result)

---
## Noninterruption

Tests two domain types: no-free (all elements have `free=n`) and single-free (`free=n` or one free element).  
Reports strict spans for complete and partial qualification.

In [ ]:
from planars import noninterruption as _nonint
show_class_reports(gc, manifest, 'noninterruption', _nonint.derive_noninterruption_domains, _nonint.format_result)

---
## Stress

Derives minimal and maximal stress domains using blocked expansion from the keystone.  
Minimal: stops before the first independently stressable position. Maximal: stops before the first obligatorily-independent position.

In [ ]:
from planars import stress as _stress
show_class_reports(gc, manifest, 'stress', _stress.derive_stress_domains, _stress.format_result)

---
## Aspiration

Derives minimal and maximal aspiration domains — mirrors the stress domain logic.

In [ ]:
from planars import aspiration as _aspiration
show_class_reports(gc, manifest, 'aspiration', _aspiration.derive_aspiration_domains, _aspiration.format_result)

---
## Domain Chart

All spans plotted as horizontal segments. Hover over a segment to see position names and span size.

In [ ]:
from planars.charts import collect_all_spans_from_sheets, charts_by_language

df, lang_meta = collect_all_spans_from_sheets(gc, manifest)
for lang_id, fig in charts_by_language(df, lang_meta).items():
    print(f"--- {lang_id} ---")
    fig.show()